# Segmentação de perfis operacionais

Execute as células em ordem.

## 1. Configuração e funções

In [1]:
"""Converte os dois clusters comportamentais em perfis operacionais de CRM.

Pré-requisito: o CSV de entrada deve ser o resultado da clusterização original
de clientes ativos e conter uma linha por id_cliente.
"""

from pathlib import Path

import numpy as np
import pandas as pd


# Ajuste os caminhos e a data de referência caso necessário.
DATA_REFERENCIA = pd.Timestamp("2026-06-30")
ARQUIVO_CLUSTER = Path("../resultados/clientes_clusterizados_ativos.csv")
ARQUIVO_CLIENTES = Path("../data/clientes.csv")
ARQUIVO_INTERACOES = Path("../data/interacoes.csv")
DIRETORIO_SAIDA = Path("../resultados/perfis_operacionais")

# No resultado original analisado, o cluster 0 corresponde aos fiéis/recorrentes
# e o cluster 1 aos compradores ocasionais. Altere apenas se seus rótulos forem
# invertidos no CSV de entrada.
CLUSTER_FIEIS = 0
CLUSTER_OCASIONAIS = 1


def normalizar_booleano(serie: pd.Series) -> pd.Series:
    """Converte TRUE/FALSE textuais ou booleanos em valores booleanos."""
    return (
        serie.astype(str)
        .str.strip()
        .str.upper()
        .map({"TRUE": True, "FALSE": False, "1": True, "0": False})
        .fillna(False)
        .astype(bool)
    )


def exigir_colunas(df: pd.DataFrame, colunas: list[str], nome: str) -> None:
    faltantes = sorted(set(colunas) - set(df.columns))
    if faltantes:
        raise ValueError(f"{nome} não contém as colunas necessárias: {faltantes}")

## 2. Carregar dados e preparar a base ativa

In [2]:
DIRETORIO_SAIDA.mkdir(parents=True, exist_ok=True)

cluster = pd.read_csv(ARQUIVO_CLUSTER)
clientes = pd.read_csv(ARQUIVO_CLIENTES)
interacoes = pd.read_csv(ARQUIVO_INTERACOES)

exigir_colunas(cluster, [
    "id_cliente", "cluster", "recencia_dias", "n_pedidos_12m",
    "ticket_medio", "variacao_frequencia_90d",
    "proporcao_pedidos_cupom", "desconto_medio_pct",
    "pct_pedidos_atrasados", "taxa_abertura_email", "taxa_cliques",
    "sessoes_site_app", "carrinhos_abandonados",
    "taxa_tickets_nao_resolvidos",
], "CSV de clusters")
exigir_colunas(clientes, ["id_cliente", "optout", "optin_email"], "clientes.csv")
exigir_colunas(interacoes, [
    "id_cliente", "ano_mes", "campanhas_reativacao_recebidas"
], "interacoes.csv")

if cluster["id_cliente"].duplicated().any():
    raise ValueError("O CSV de clusters possui id_cliente duplicado.")

# O CSV de clusters já deve conter somente clientes ativos. Esta filtragem
# garante aderência à regra de negócio caso o arquivo tenha linhas extras.
segmentacao = cluster[cluster["recencia_dias"] <= 180].copy()

# Evita duplicar colunas caso o CSV de clusters já guarde dados cadastrais.
segmentacao = segmentacao.drop(columns=["optout", "optin_email"], errors="ignore")
cadastro = clientes[["id_cliente", "optout", "optin_email"]].copy()
cadastro["optout"] = normalizar_booleano(cadastro["optout"])
cadastro["optin_email"] = normalizar_booleano(cadastro["optin_email"])

# Soma as campanhas de reativação nos três meses mais recentes disponíveis.
interacoes["data_mes"] = pd.to_datetime(
    interacoes["ano_mes"].astype(str) + "-01", errors="coerce"
)
inicio_90_dias = DATA_REFERENCIA - pd.DateOffset(months=3)
campanhas_90d = (
    interacoes.loc[
        interacoes["data_mes"].between(inicio_90_dias, DATA_REFERENCIA),
        ["id_cliente", "campanhas_reativacao_recebidas"],
    ]
    .groupby("id_cliente", as_index=False)
    .agg(campanhas_reativacao_90d=("campanhas_reativacao_recebidas", "sum"))
)

segmentacao = (
    segmentacao
    .merge(cadastro, on="id_cliente", how="left")
    .merge(campanhas_90d, on="id_cliente", how="left")
)
segmentacao["optout"] = segmentacao["optout"].fillna(False).astype(bool)
segmentacao["optin_email"] = segmentacao["optin_email"].fillna(False).astype(bool)
segmentacao["campanhas_reativacao_90d"] = (
    pd.to_numeric(segmentacao["campanhas_reativacao_90d"], errors="coerce")
    .fillna(0)
)
segmentacao["elegivel_campanha"] = ~segmentacao["optout"]

# Garante que as métricas usadas nas regras sejam numéricas.
metricas = [
    "recencia_dias", "ticket_medio",
    "pct_pedidos_atrasados", "taxa_tickets_nao_resolvidos",
    "taxa_abertura_email", "taxa_cliques", "sessoes_site_app",
    "carrinhos_abandonados", "proporcao_pedidos_cupom",
    "desconto_medio_pct", "variacao_frequencia_90d",
]
for coluna in metricas:
    segmentacao[coluna] = pd.to_numeric(segmentacao[coluna], errors="coerce").fillna(0)

cluster_1 = segmentacao.loc[
    segmentacao["cluster"] == CLUSTER_OCASIONAIS
].copy()
if cluster_1.empty:
    raise ValueError(
        f"Não há clientes no cluster {CLUSTER_OCASIONAIS}. "
        "Confira os valores de CLUSTER_FIEIS e CLUSTER_OCASIONAIS."
    )

## 3. Calcular limiares do Cluster 1

In [3]:
# Os limiares são relativos ao Cluster 1, evitando impor valores arbitrários.

p60_sessoes = cluster_1["sessoes_site_app"].quantile(0.60)
p60_carrinhos = cluster_1["carrinhos_abandonados"].quantile(0.60)
p60_cupom = cluster_1["proporcao_pedidos_cupom"].quantile(0.60)
p60_desconto = cluster_1["desconto_medio_pct"].quantile(0.60)
p75_campanhas = cluster_1["campanhas_reativacao_90d"].quantile(0.75)
p90_atrasos = cluster_1["pct_pedidos_atrasados"].quantile(0.90)

## 4. Aplicar perfis e ações

In [4]:
# Perfil-base definido pela clusterização original.

segmentacao["perfil_final"] = np.select(
    [
        segmentacao["cluster"] == CLUSTER_FIEIS,
        segmentacao["cluster"] == CLUSTER_OCASIONAIS,
    ],
    [
        "Fiéis em crescimento",
        "Compradores ocasionais de baixa prioridade",
    ],
    default="Cluster não mapeado",
)
segmentacao["acao_recomendada"] = np.select(
    [
        segmentacao["cluster"] == CLUSTER_FIEIS,
        segmentacao["cluster"] == CLUSTER_OCASIONAIS,
    ],
    [
        "Relacionamento e benefícios do clube; evitar cupom automático",
        "Acompanhar; não priorizar neste ciclo",
    ],
    default="Revisar classificação do cluster",
)

# A ordem abaixo é deliberada: serviço é tratado antes de marketing; fadiga
# também prevalece sobre uma oferta, pois insistir aumenta risco de descadastro.
mask_friccao = (
    (segmentacao["taxa_tickets_nao_resolvidos"] > 0)
    | (segmentacao["pct_pedidos_atrasados"] >= p90_atrasos)
)
segmentacao.loc[mask_friccao, "perfil_final"] = "Clientes com fricção de serviço"
segmentacao.loc[mask_friccao, "acao_recomendada"] = (
    "Resolver atraso ou SAC antes de oferecer benefício"
)

perfil_padrao_c1 = "Compradores ocasionais de baixa prioridade"
mask_saturados = (
    (segmentacao["cluster"] == CLUSTER_OCASIONAIS)
    & (segmentacao["perfil_final"] == perfil_padrao_c1)
    & (segmentacao["campanhas_reativacao_90d"] >= p75_campanhas)
    & (segmentacao["taxa_abertura_email"] == 0)
    & (segmentacao["taxa_cliques"] == 0)
)
segmentacao.loc[mask_saturados, "perfil_final"] = "Saturados de comunicação"
segmentacao.loc[mask_saturados, "acao_recomendada"] = (
    "Pausar comunicação por um ciclo ou reduzir frequência"
)

mask_digitais = (
    (segmentacao["cluster"] == CLUSTER_OCASIONAIS)
    & (segmentacao["perfil_final"] == perfil_padrao_c1)
    & (
        (segmentacao["sessoes_site_app"] >= p60_sessoes)
        | (segmentacao["carrinhos_abandonados"] >= p60_carrinhos)
    )
    & (
        (segmentacao["taxa_abertura_email"] > 0)
        | (segmentacao["taxa_cliques"] > 0)
    )
)
segmentacao.loc[mask_digitais, "perfil_final"] = "Recuperáveis com intenção digital"
segmentacao.loc[mask_digitais, "acao_recomendada"] = (
    "Lembrete de carrinho, recomendação ou benefício leve"
)

mask_promocao = (
    (segmentacao["cluster"] == CLUSTER_OCASIONAIS)
    & (segmentacao["perfil_final"] == perfil_padrao_c1)
    & (segmentacao["proporcao_pedidos_cupom"] >= p60_cupom)
    & (segmentacao["desconto_medio_pct"] >= p60_desconto)
    & (segmentacao["variacao_frequencia_90d"] <= 0)
)
segmentacao.loc[mask_promocao, "perfil_final"] = (
    "Sensíveis a promoção em desaceleração"
)
segmentacao.loc[mask_promocao, "acao_recomendada"] = (
    "Kit, frete ou benefício moderado; evitar cupom de 25%"
)

# Opt-out não altera o perfil comportamental, mas sempre bloqueia o contato.
segmentacao.loc[~segmentacao["elegivel_campanha"], "acao_recomendada"] = (
    "Não contatar: cliente com opt-out"
)

## 5. Calcular score de urgência e selecionar os 1.500 contatos

In [5]:
# A capacidade da campanha é definida por prioridade, e não por uma cota

# fixa para cada perfil. Os perfis indicam *como* abordar o cliente; o
# score indica *quem* deve receber o contato neste ciclo.
perfis_excluidos_da_priorizacao = [
    "Fiéis em crescimento",
    "Saturados de comunicação",
]
candidatos = segmentacao.loc[
    segmentacao["elegivel_campanha"]
    & ~segmentacao["perfil_final"].isin(perfis_excluidos_da_priorizacao)
].copy()

# Cada componente é transformado em percentil somente entre os candidatos
# elegíveis. Assim, escalas diferentes (dias, reais e contagens) não
# distorcem o resultado. Frequência em queda recebe maior prioridade.
candidatos["score_recencia"] = candidatos["recencia_dias"].rank(pct=True)
candidatos["score_queda_frequencia"] = (
    -candidatos["variacao_frequencia_90d"]
).rank(pct=True)
candidatos["score_intencao_digital"] = (
    0.50 * candidatos["sessoes_site_app"].rank(pct=True)
    + 0.30 * candidatos["carrinhos_abandonados"].rank(pct=True)
    + 0.10 * candidatos["taxa_abertura_email"].rank(pct=True)
    + 0.10 * candidatos["taxa_cliques"].rank(pct=True)
)
candidatos["score_valor_economico"] = candidatos["ticket_medio"].rank(pct=True)
candidatos["score_urgencia"] = (
    0.40 * candidatos["score_recencia"]
    + 0.25 * candidatos["score_queda_frequencia"]
    + 0.20 * candidatos["score_intencao_digital"]
    + 0.15 * candidatos["score_valor_economico"]
)

# Desempates previsíveis: maior recência e, por fim, id do cliente.
top_1500 = (
    candidatos
    .sort_values(
        ["score_urgencia", "recencia_dias", "id_cliente"],
        ascending=[False, False, True],
    )
    .head(1500)
    [["id_cliente", "perfil_final", "score_urgencia"]]
    .copy()
)
top_1500["score_urgencia"] = top_1500["score_urgencia"].round(6)

# Mantém o score e o indicador de seleção também no arquivo completo.
segmentacao = segmentacao.merge(
    top_1500[["id_cliente", "score_urgencia"]], on="id_cliente", how="left"
)
segmentacao["selecionado_proximo_ciclo"] = segmentacao["score_urgencia"].notna()

## 6. Gerar resumo e salvar CSVs

In [6]:
resumo = (
    segmentacao.groupby("perfil_final", as_index=False)
    .agg(
        clientes=("id_cliente", "count"),
        elegiveis_campanha=("elegivel_campanha", "sum"),
        optouts=("optout", "sum"),
        recencia_media=("recencia_dias", "mean"),
        pedidos_medios=("n_pedidos_12m", "mean"),
        ticket_medio=("ticket_medio", "mean"),
        cupom_medio=("proporcao_pedidos_cupom", "mean"),
        abertura_email_media=("taxa_abertura_email", "mean"),
        atraso_medio=("pct_pedidos_atrasados", "mean"),
        selecionados_proximo_ciclo=("selecionado_proximo_ciclo", "sum"),
    )
)
resumo["pct_optout"] = (100 * resumo["optouts"] / resumo["clientes"]).round(2)
resumo = resumo.sort_values("clientes", ascending=False)

caminho_clientes = DIRETORIO_SAIDA / "clientes_segmentados_por_perfil.csv"
caminho_resumo = DIRETORIO_SAIDA / "resumo_perfis_operacionais.csv"
caminho_top_1500 = DIRETORIO_SAIDA / "top_1500_priorizados.csv"
segmentacao.to_csv(caminho_clientes, index=False, encoding="utf-8-sig")
resumo.to_csv(caminho_resumo, index=False, encoding="utf-8-sig")
top_1500.to_csv(caminho_top_1500, index=False, encoding="utf-8-sig")

print("Segmentação concluída.")
print(f"Clientes ativos analisados: {len(segmentacao):,}")
print(f"Clientes elegíveis para campanha: {segmentacao['elegivel_campanha'].sum():,}")
print(f"Selecionados para o próximo ciclo: {len(top_1500):,}")
print("\nQuantidade por perfil:")
print(segmentacao["perfil_final"].value_counts().to_string())
print(f"\nArquivo de clientes: {caminho_clientes}")
print(f"Resumo dos perfis: {caminho_resumo}")
print(f"Top 1.500 priorizados: {caminho_top_1500}")

Segmentação concluída.
Clientes ativos analisados: 6,289
Clientes elegíveis para campanha: 6,064
Selecionados para o próximo ciclo: 1,500

Quantidade por perfil:
perfil_final
Fiéis em crescimento                          2390
Clientes com fricção de serviço               1907
Recuperáveis com intenção digital             1135
Compradores ocasionais de baixa prioridade     584
Sensíveis a promoção em desaceleração          260
Saturados de comunicação                        13

Arquivo de clientes: ../resultados/perfis_operacionais/clientes_segmentados_por_perfil.csv
Resumo dos perfis: ../resultados/perfis_operacionais/resumo_perfis_operacionais.csv
Top 1.500 priorizados: ../resultados/perfis_operacionais/top_1500_priorizados.csv
